In [1]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt
import scipy as scp
import multiprocessing

/Users/andpetrouzeniou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def c_quantile(a, Sigma_Y, n_iter, rand):

    N = len(Sigma_Y)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_Y, n_iter)
    diags = np.tile((1 / np.diagonal(Sigma_Y) ** (1/2)), (n_iter, 1))
    
    draws = abs(draws * diags)
    
    lst = np.amax(draws, axis=1)
    return np.quantile(np.array(lst), a)


In [3]:
def d_quantile(b, k, Sigma_X, n_iter, rand):

    N = len(Sigma_X)
    
    draws = rand.multivariate_normal(np.zeros(N), Sigma_X, n_iter)
    lst = np.amax(draws, axis=1) - np.amax((-1 * draws), axis=1)

    return np.quantile(np.array(lst), b)


In [4]:
def make_I_list(rank_list, X_temp, delta_L, delta_U):
    
    rank = np.max(np.array(rank_list))
    
    # need matrix A[i,j] = X_temp[j] + delta_L[j,i]
    
    X_temp_mat = np.tile(X_temp, (len(X_temp), 1))
    A = X_temp_mat + np.transpose(delta_L)
    
    np.fill_diagonal(A, np.nan)
    A = A[~np.isnan(A)].reshape(A.shape[0], A.shape[1] - 1)

    row_ranks = np.partition(A, -rank, axis=1)[:, -rank]
    
    indices = np.where(X_temp >= row_ranks, 1, 0)
        
    return indices


In [5]:
def c_2_quantile(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):

    lst = []

    Sigma_Y = Sigma

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        rank = np.max(np.array(rank_list))

        I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)
        
        I_indices = np.where(I_list == 1)[0]    
        lst.append(np.max((abs(X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))
    
#         s = []
                                        
#         for i in range(len(rank_list)):
                                        
#             I = I_list[i]
#             I_indices = np.where(I == 1)[0]
            
#             s.append(np.max((abs(X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))

    return np.quantile(np.array(lst), ab)


In [6]:
def c_2_quantile_asymmetric(ab, rank_list, Sigma, delta_L, delta_U, n_iter, rand):

    ab_p = 1 - ab
    ab_p = ab_p / 2
    ab_p = 1 - ab_p
    
    lst_l = []
    lst_r = []

    Sigma_Y = Sigma

    for a in range(n_iter):
        print(a, end="")
        # print(i, end="")
        draw = rand.multivariate_normal(np.zeros(len(Sigma)), Sigma)
            
        X_temp = draw.copy()
        rank = np.max(np.array(rank_list))

        I_list = make_I_list(rank_list, X_temp, delta_L, delta_U)
        
        I_indices = np.where(I_list == 1)[0]    
        lst_r.append(np.max(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))
        lst_l.append(np.min(((X_temp) / (np.diagonal(Sigma_Y) ** 0.5))[I_indices]))

    return([np.quantile(-1 * np.array(lst_l), ab_p), np.quantile(np.array(lst_r), ab_p)])


In [7]:
def c_3_quantile(ab, subset, Sigma, n_iter, rand):
    
    lst = []
    N = len(Sigma)
    
    for i in range(n_iter):
        draw = rand.multivariate_normal(np.zeros(N), Sigma)
        
        draw = draw[subset]
        diag = np.diagonal(Sigma)[subset]
        lst.append(np.max(abs(draw / (diag ** (1/2)))))
            
    return np.quantile(np.array(lst), ab)


In [8]:
# projection CI

def CI_proj(rank_list, y_vec, a, Sigma_Y, n_iter, quantile_dict, rand):

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])
        
    try:
        c_a = quantile_dict[str(a) + str(Sigma_Y) + "c"]
    except:
        c_a = c_quantile(a, Sigma_Y, n_iter, rand = rand)
        quantile_dict[str(a) + str(Sigma_Y) + "c"] = c_a
        
    CI_list = []
        
    for i in range(len(rank_list)):  

        CI_lower_bound = y_vec[th_hat_lst[i]] - c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + c_a * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


In [9]:
def two_step_CI(rank_list, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    k = len(y_vec)

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])

    d = d_quantile(1 - b, k, Sigma_Y, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    L = r - l - d
    U = r - l + d
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile(1-a+b, rank_list, Sigma_Y, L, U, int(n_iter), rand = rand)
    print()
    
    CI_list = []
    
    for i in range(len(rank_list)):
        th_hat = th_hat_lst[i]
        CI_t = [y_vec[th_hat] - bd * (Sigma_Y[th_hat, th_hat] ** 0.5), y_vec[th_hat] + bd * (Sigma_Y[th_hat, th_hat] ** 0.5)]
        CI_list.append(CI_t)

    return CI_list


In [10]:
def two_step_CI_asymmetric(rank_list, y_vec, a, b, Sigma, n_iter, quantile_dict, rand):
    
    Sigma_Y = Sigma
    k = len(y_vec)

    th_hat_lst = [] 
    
    for i in range(len(rank_list)):
        th_hat_lst.append(np.argsort(y_vec)[len(y_vec)-rank_list[i]])

    d = d_quantile(1 - b, k, Sigma_Y, n_iter, rand = rand)
    print()
    
    l = np.tile(y_vec, (len(y_vec), 1))
    r = np.transpose(l)
    
    L = r - l - d
    U = r - l + d
    
    # L[1,2] = r[1,2] - l[1,2] - d = l[2,1] - l[1,2] - d = y_vec[1] - y_vec[2] - d
    
    bd = c_2_quantile_asymmetric(1 - a + b, rank_list, Sigma, L, U, int(n_iter), rand = rand)
    bd_l = bd[0]
    bd_r = bd[1]
    print()
    
    CI_list = []
    
    for i in range(len(rank_list)):
        
        CI_lower_bound = y_vec[th_hat_lst[i]] - bd_r * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        CI_upper_bound = y_vec[th_hat_lst[i]] + bd_l * (Sigma_Y[th_hat_lst[i], th_hat_lst[i]] ** 0.5)
        
        CI_list.append([CI_lower_bound, CI_upper_bound])

    return CI_list


# Get CSs for all top 50 CZs

In [ ]:
cz_list = ["Port St. Lucie", "Chicago", "Philadelphia", "New York", "Newark", "Milwaukee", "Atlanta", "St. Louis", "Cleveland", "Miami", "Detroit", "Pittsburgh", "Columbus", "Bridgeport", "Dallas", "Washington DC", "Austin", "Raleigh", "Cincinnati", "Charlotte", "Kansas City", "New Orleans", "Nashville", "Baltimore", "Buffalo", "Indianapolis", "Grand Rapids", "Dayton", "San Francisco", "Tampa", "Salt Lake City", "Boston", "Houston", "Phoenix", "Minneapolis", "Denver", "Los Angeles", "Jacksonville", "Fort Worth", "San Antonio", "Orlando", "San Jose", "Sacramento", "San Diego", "Manchester", "Providence", "Fresno", "Las Vegas", "Portland", "Seattle"]

In [ ]:
def make_CIs(cz):

    Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy()
    X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy().flatten()
    Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_"+cz+".csv",index_col = 0).to_numpy()).flatten()
    rank_list = [i for i in range(1,int(len(Y) / 3))]

    print("Projection")
    proj = pd.DataFrame(CI_proj(rank_list, Y, 0.95, Sigma, 10000, {}, rng))

    print("Two Step")
    two_step = pd.DataFrame(two_step_CI(rank_list, Y, 0.05, 0.005, Sigma, 10000, {}, rng))
    
    print("Two Step Asymmetric")
    two_step_asymmetric = pd.DataFrame(two_step_CI_asymmetric(rank_list, Y, 0.05, 0.005, Sigma, 10~000, {}, rng))
    
    proj.to_csv("All_CIs/Projection_"+cz+".csv")
    two_step.to_csv("All_CIs/TwoStep_"+cz+".csv")
    two_step_asymmetric.to_csv("All_CIs/TwoStepAsymmetric_"+cz+".csv")
    
    

In [ ]:
rng = np.random.default_rng(42)

pool = multiprocessing.Pool(55)
p_cov_prob, shaikh_cov_prob, p_length, shaikh_length = zip(*pool.map(make_CIs, cz_list))

In [13]:
import os

notebook_path = os.getcwd()
print(notebook_path)

/Users/andpetrouzeniou/Documents/GitHub/Inference_on_Multiple_Winners/Neighborhood_Effects_Application


# Seattle and Cleveland Urban CZs

In [11]:

Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy()
X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy().flatten()
Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_Seattle_urban.csv",index_col = 0).to_numpy()).flatten()


In [12]:
print(len(Sigma))

132


In [13]:
rng = np.random.default_rng(42)

print("Two Step")
two_step_sea = two_step_CI([i for i in range(1,int(len(Y) / 5))], Y, 0.05, 0.005, Sigma, 10000, {}, rng)

print("Projection")
proj_sea = CI_proj([i for i in range(1,int(len(Y) / 5))], Y, 0.95, Sigma, 10000, {}, rng)
    

Two Step



0123456789101112131415161718192021222324252627282930313233343536373839404142434445464748495051525354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899100101102103104105106107108109110111112113114115116117118119120121122123124125126127128129130131132133134135136137138139140141142143144145146147148149150151152153154155156157158159160161162163164165166167168169170171172173174175176177178179180181182183184185186187188189190191192193194195196197198199200201202203204205206207208209210211212213214215216217218219220221222223224225226227228229230231232233234235236237238239240241242243244245246247248249250251252253254255256257258259260261262263264265266267268269270271272273274275276277278279280281282283284285286287288289290291292293294295296297298299300301302303304305306307308309310311312313314315316317318319320321322323324325326327328329330331332333334335336337338339340341342343344345346347348349350351352353354355356357358359360361362363364365366367368369

2405240624072408240924102411241224132414241524162417241824192420242124222423242424252426242724282429243024312432243324342435243624372438243924402441244224432444244524462447244824492450245124522453245424552456245724582459246024612462246324642465246624672468246924702471247224732474247524762477247824792480248124822483248424852486248724882489249024912492249324942495249624972498249925002501250225032504250525062507250825092510251125122513251425152516251725182519252025212522252325242525252625272528252925302531253225332534253525362537253825392540254125422543254425452546254725482549255025512552255325542555255625572558255925602561256225632564256525662567256825692570257125722573257425752576257725782579258025812582258325842585258625872588258925902591259225932594259525962597259825992600260126022603260426052606260726082609261026112612261326142615261626172618261926202621262226232624262526262627262826292630263126322633263426352636263726382639264026412642264326442645264626472648264926502651265226532654

4474447544764477447844794480448144824483448444854486448744884489449044914492449344944495449644974498449945004501450245034504450545064507450845094510451145124513451445154516451745184519452045214522452345244525452645274528452945304531453245334534453545364537453845394540454145424543454445454546454745484549455045514552455345544555455645574558455945604561456245634564456545664567456845694570457145724573457445754576457745784579458045814582458345844585458645874588458945904591459245934594459545964597459845994600460146024603460446054606460746084609461046114612461346144615461646174618461946204621462246234624462546264627462846294630463146324633463446354636463746384639464046414642464346444645464646474648464946504651465246534654465546564657465846594660466146624663466446654666466746684669467046714672467346744675467646774678467946804681468246834684468546864687468846894690469146924693469446954696469746984699470047014702470347044705470647074708470947104711471247134714471547164717471847194720472147224723

6549655065516552655365546555655665576558655965606561656265636564656565666567656865696570657165726573657465756576657765786579658065816582658365846585658665876588658965906591659265936594659565966597659865996600660166026603660466056606660766086609661066116612661366146615661666176618661966206621662266236624662566266627662866296630663166326633663466356636663766386639664066416642664366446645664666476648664966506651665266536654665566566657665866596660666166626663666466656666666766686669667066716672667366746675667666776678667966806681668266836684668566866687668866896690669166926693669466956696669766986699670067016702670367046705670667076708670967106711671267136714671567166717671867196720672167226723672467256726672767286729673067316732673367346735673667376738673967406741674267436744674567466747674867496750675167526753675467556756675767586759676067616762676367646765676667676768676967706771677267736774677567766777677867796780678167826783678467856786678767886789679067916792679367946795679667976798

8611861286138614861586168617861886198620862186228623862486258626862786288629863086318632863386348635863686378638863986408641864286438644864586468647864886498650865186528653865486558656865786588659866086618662866386648665866686678668866986708671867286738674867586768677867886798680868186828683868486858686868786888689869086918692869386948695869686978698869987008701870287038704870587068707870887098710871187128713871487158716871787188719872087218722872387248725872687278728872987308731873287338734873587368737873887398740874187428743874487458746874787488749875087518752875387548755875687578758875987608761876287638764876587668767876887698770877187728773877487758776877787788779878087818782878387848785878687878788878987908791879287938794879587968797879887998800880188028803880488058806880788088809881088118812881388148815881688178818881988208821882288238824882588268827882888298830883188328833883488358836883788388839884088418842884388448845884688478848884988508851885288538854885588568857885888598860

In [14]:
print("Projection")
print(proj_sea)
print("Two Step")
print(two_step_sea)

Projection
[[0.03261455993587706, 0.31322925058289297], [-0.008284305095014877, 0.32519971561378286], [-0.028476475830470915, 0.28833402634924093], [-0.05451126353307112, 0.30171557405184113], [-0.034632698692426744, 0.27333214921119675], [-0.028504570654808478, 0.25575480117357846], [-0.08613933920908594, 0.2883543097278559], [-0.03674390655616934, 0.23660245707493854], [-0.04200816847038269, 0.23637213898915188], [-0.032456039567075795, 0.222188490085845], [-0.03066122738902434, 0.21589553790779337], [-0.07514704171334777, 0.25971939223211715], [-0.03877866411154654, 0.22324205463031574], [-0.06543874169345297, 0.24717689221222194], [-0.2507141031334717, 0.4319960336522409], [-0.08580261000681583, 0.256727400525585], [-0.01322321380925319, 0.1804537043280226], [-0.029518529579816113, 0.19653778009858552], [-0.1189025461444038, 0.2783300166631732], [-0.04891316526668514, 0.20556229578545454], [0.01415807768774427, 0.14186997283102512], [-0.08113655763987448, 0.2350539481586439], [-0.0

In [19]:

Sigma = pd.read_csv("Input_Data_Processed_Urban/Sigma_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()
X = pd.read_csv("Input_Data_Processed_Urban/X_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy().flatten()
Y = np.array(pd.read_csv("Input_Data_Processed_Urban/Y_quantiles_neighborhood_Cleveland_urban.csv",index_col = 0).to_numpy()).flatten()


In [20]:
rng = np.random.default_rng(90)

print("Projection")
proj_cle = CI_proj([i for i in range(1,int(len(Y) / 5))], Y, 0.95, Sigma, 10000, {}, rng)

print("Two Step")
two_step_cle = two_step_CI([i for i in range(1,int(len(Y) / 5))], Y, 0.05, 0.005, Sigma, 10000, {}, rng)
    

Projection
Two Step



0123456789101112131415161718192021222324252627282930313233343536373839404142434445464748495051525354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899100101102103104105106107108109110111112113114115116117118119120121122123124125126127128129130131132133134135136137138139140141142143144145146147148149150151152153154155156157158159160161162163164165166167168169170171172173174175176177178179180181182183184185186187188189190191192193194195196197198199200201202203204205206207208209210211212213214215216217218219220221222223224225226227228229230231232233234235236237238239240241242243244245246247248249250251252253254255256257258259260261262263264265266267268269270271272273274275276277278279280281282283284285286287288289290291292293294295296297298299300301302303304305306307308309310311312313314315316317318319320321322323324325326327328329330331332333334335336337338339340341342343344345346347348349350351352353354355356357358359360361362363364365366367368369

2346234723482349235023512352235323542355235623572358235923602361236223632364236523662367236823692370237123722373237423752376237723782379238023812382238323842385238623872388238923902391239223932394239523962397239823992400240124022403240424052406240724082409241024112412241324142415241624172418241924202421242224232424242524262427242824292430243124322433243424352436243724382439244024412442244324442445244624472448244924502451245224532454245524562457245824592460246124622463246424652466246724682469247024712472247324742475247624772478247924802481248224832484248524862487248824892490249124922493249424952496249724982499250025012502250325042505250625072508250925102511251225132514251525162517251825192520252125222523252425252526252725282529253025312532253325342535253625372538253925402541254225432544254525462547254825492550255125522553255425552556255725582559256025612562256325642565256625672568256925702571257225732574257525762577257825792580258125822583258425852586258725882589259025912592259325942595

4406440744084409441044114412441344144415441644174418441944204421442244234424442544264427442844294430443144324433443444354436443744384439444044414442444344444445444644474448444944504451445244534454445544564457445844594460446144624463446444654466446744684469447044714472447344744475447644774478447944804481448244834484448544864487448844894490449144924493449444954496449744984499450045014502450345044505450645074508450945104511451245134514451545164517451845194520452145224523452445254526452745284529453045314532453345344535453645374538453945404541454245434544454545464547454845494550455145524553455445554556455745584559456045614562456345644565456645674568456945704571457245734574457545764577457845794580458145824583458445854586458745884589459045914592459345944595459645974598459946004601460246034604460546064607460846094610461146124613461446154616461746184619462046214622462346244625462646274628462946304631463246334634463546364637463846394640464146424643464446454646464746484649465046514652465346544655

6472647364746475647664776478647964806481648264836484648564866487648864896490649164926493649464956496649764986499650065016502650365046505650665076508650965106511651265136514651565166517651865196520652165226523652465256526652765286529653065316532653365346535653665376538653965406541654265436544654565466547654865496550655165526553655465556556655765586559656065616562656365646565656665676568656965706571657265736574657565766577657865796580658165826583658465856586658765886589659065916592659365946595659665976598659966006601660266036604660566066607660866096610661166126613661466156616661766186619662066216622662366246625662666276628662966306631663266336634663566366637663866396640664166426643664466456646664766486649665066516652665366546655665666576658665966606661666266636664666566666667666866696670667166726673667466756676667766786679668066816682668366846685668666876688668966906691669266936694669566966697669866996700670167026703670467056706670767086709671067116712671367146715671667176718671967206721

8544854585468547854885498550855185528553855485558556855785588559856085618562856385648565856685678568856985708571857285738574857585768577857885798580858185828583858485858586858785888589859085918592859385948595859685978598859986008601860286038604860586068607860886098610861186128613861486158616861786188619862086218622862386248625862686278628862986308631863286338634863586368637863886398640864186428643864486458646864786488649865086518652865386548655865686578658865986608661866286638664866586668667866886698670867186728673867486758676867786788679868086818682868386848685868686878688868986908691869286938694869586968697869886998700870187028703870487058706870787088709871087118712871387148715871687178718871987208721872287238724872587268727872887298730873187328733873487358736873787388739874087418742874387448745874687478748874987508751875287538754875587568757875887598760876187628763876487658766876787688769877087718772877387748775877687778778877987808781878287838784878587868787878887898790879187928793

In [21]:
print("Projection")
print(proj_cle)
print("Two Step")
print(two_step_cle)

Projection
[[0.08030950383521288, 0.3625011202649351], [0.06240709012343705, 0.284094953976711], [0.06993370751292988, 0.27484885658721814], [0.03837019661991049, 0.2523786074802375], [0.03232501727576985, 0.23222450682437812], [0.052966474134494346, 0.20600214996565364], [0.04312065246325612, 0.21162391163689187], [0.036746973058435356, 0.21012023104171265], [0.03967852809384144, 0.19558883600630655], [-0.019491017814256745, 0.23691996191440476], [-0.008680965440237332, 0.21373718954038534], [0.01556605612425506, 0.17840422797589234], [0.012832646602764314, 0.16881331749738268], [-0.007288001536680411, 0.1885959456368276], [-0.0826529039364722, 0.25036312803661964], [-0.013809252984092676, 0.17969479708424008], [-0.06283038216013052, 0.2274221462602779], [0.013396308788915803, 0.15086237531123142], [0.018658024211530122, 0.1446767998886171], [0.00992046544899608, 0.1466124986511513], [0.0028683963832094456, 0.14732764771693796], [-0.027276500471462503, 0.1610272045716099], [-0.0025644

In [22]:
pd.DataFrame(proj_cle).to_csv("Urban_CIs/CI_Cleveland_proj.csv")
pd.DataFrame(proj_sea).to_csv("Urban_CIs/CI_Seattle_proj.csv")

pd.DataFrame(two_step_cle).to_csv("Urban_CIs/CI_Cleveland_two_step.csv")
pd.DataFrame(two_step_sea).to_csv("Urban_CIs/CI_Seattle_two_step.csv")